In [18]:
from pathlib import Path

data_dir = Path("./update_data")

if data_dir.exists():
    print(f"Contents of '{data_dir.resolve()}':")
    # List all files (including subdirectories)
    all_files = list(data_dir.rglob("*"))
    
    if all_files:
        for file in all_files[:20]:  # Show first 20 items
            print(f" - {file.relative_to(data_dir)} (is_file: {file.is_file()})")
        if len(all_files) > 20:
            print(f"... and {len(all_files) - 20} more items.")
    else:
        print("The 'update_data' folder is completely empty.")
else:
    print(f"Directory '{data_dir}' does not exist.")
    

Contents of 'C:\Users\yempa\OneDrive\Attachments\Documents\New folder\Data_visualisation\update_data':
 - 2026_08_17T22_10.csv (is_file: True)
 - 2026_08_17T22_15.csv (is_file: True)
 - 2026_08_17T22_20.csv (is_file: True)
 - 2026_08_17T22_25.csv (is_file: True)
 - 2026_08_17T22_30.csv (is_file: True)
 - 2026_08_17T22_35.csv (is_file: True)
 - 2026_08_17T22_40.csv (is_file: True)
 - 2026_08_17T22_45.csv (is_file: True)
 - 2026_08_17T22_50.csv (is_file: True)
 - 2026_08_17T22_55.csv (is_file: True)
 - 2026_08_17T23_00.csv (is_file: True)
 - 2026_08_17T23_05.csv (is_file: True)
 - 2026_08_17T23_10.csv (is_file: True)
 - 2026_08_17T23_15.csv (is_file: True)
 - 2026_08_17T23_20.csv (is_file: True)
 - 2026_08_17T23_25.csv (is_file: True)
 - 2026_08_17T23_30.csv (is_file: True)
 - 2026_08_17T23_35.csv (is_file: True)
 - 2026_08_17T23_40.csv (is_file: True)
 - 2026_08_17T23_45.csv (is_file: True)
... and 4034 more items.


In [19]:
from pathlib import Path
import polars as pl

data_dir = Path("./update_data")
path_pattern = str(data_dir / "*.csv")

print(f"Scanning CSV files from: {path_pattern}")

# 1. Let's first inspect the headers of a single file to confirm column names
sample_file = next(data_dir.glob("*.csv"))
print(f"\nInspecting headers of sample file: {sample_file.name}")
schema_df = pl.read_csv(sample_file, n_rows=5)
print("Available columns:", schema_df.columns)
print(schema_df.head(2))

# 2. Build the full lazy pipeline using *.csv
lf = pl.scan_csv(
    path_pattern,
    infer_schema_length=10000,
    ignore_errors=True,
)

# Note: Adjust column names below if your CSV columns differ slightly from 'arrival_delay', 'departure_delay', etc.
print("\nCompiling lazy pipeline...")
cleaned_lf = (
    lf
    .with_columns([
        # If your files contain a snapshot timestamp column or you extract it from the filename, 
        # adapt accordingly. Let's inspect a sample first to see how timestamps are stored.
    ])
)

Scanning CSV files from: update_data\*.csv

Inspecting headers of sample file: 2026_08_17T22_10.csv
Available columns: ['trip_id', 'stop_id', 'delay_seconds', 'timestamp_fetched']
shape: (2, 4)
┌─────────┬─────────┬───────────────┬────────────────────────────┐
│ trip_id ┆ stop_id ┆ delay_seconds ┆ timestamp_fetched          │
│ ---     ┆ ---     ┆ ---           ┆ ---                        │
│ i64     ┆ i64     ┆ i64           ┆ str                        │
╞═════════╪═════════╪═══════════════╪════════════════════════════╡
│ 1041819 ┆ 672646  ┆ 660           ┆ 2026-08-17T22:10:00.000000 │
│ 1414685 ┆ 604323  ┆ 68            ┆ 2026-08-17T22:10:00.000000 │
└─────────┴─────────┴───────────────┴────────────────────────────┘

Compiling lazy pipeline...


In [20]:
from pathlib import Path
import polars as pl

def build_transit_pipeline(data_dir: str = "./update_data") -> pl.LazyFrame:
    """
    Scans and processes all Bremen public transport delay snapshot CSVs.
    """
    path_pattern = str(Path(data_dir) / "*.csv")
    print(f"Scanning CSV files from: {path_pattern}")
    
    # Lazy scan handles all 4,000+ files efficiently using streaming/chunked execution
    lf = pl.scan_csv(
        path_pattern,
        infer_schema_length=10000,
        ignore_errors=True,
    )
    
    # Clean and parse the schema
    cleaned_lf = (
        lf
        .with_columns([
            # Parse timestamp_fetched into proper datetime objects
            pl.col("timestamp_fetched").str.to_datetime("%Y-%m-%dT%H:%M:%S.%f", strict=False).alias("parsed_timestamp"),
            # Ensure delay_seconds is numeric, filling nulls with 0
            pl.col("delay_seconds").fill_null(0).cast(pl.Int32),
        ])
        .filter(
            pl.col("parsed_timestamp").is_not_null() & 
            pl.col("trip_id").is_not_null()
        )
    )
    
    return cleaned_lf

if __name__ == "__main__":
    lazy_transit_df = build_transit_pipeline()
    
    print("Pipeline built successfully. Collecting sample data...")
    sample_result = lazy_transit_df.head(5).collect()
    print(sample_result)
    

Scanning CSV files from: update_data\*.csv
Pipeline built successfully. Collecting sample data...


C:\Users\yempa\AppData\Local\Temp\ipykernel_31708\3459283010.py:23: ChronoFormatWarning:

Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime

C:\Users\yempa\AppData\Local\Temp\ipykernel_31708\3459283010.py:23: ChronoFormatWarning:

Detected the pattern `%f` in the chrono format string. This pattern should not be used to parse microseconds. Instead, use e.g. `%3f` for decimal fraction of a second with a fixed length of 3. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime



shape: (5, 5)
┌─────────┬─────────┬───────────────┬────────────────────────────┬─────────────────────┐
│ trip_id ┆ stop_id ┆ delay_seconds ┆ timestamp_fetched          ┆ parsed_timestamp    │
│ ---     ┆ ---     ┆ ---           ┆ ---                        ┆ ---                 │
│ i64     ┆ i64     ┆ i32           ┆ str                        ┆ datetime[μs]        │
╞═════════╪═════════╪═══════════════╪════════════════════════════╪═════════════════════╡
│ 1041819 ┆ 672646  ┆ 660           ┆ 2026-08-17T22:10:00.000000 ┆ 2026-08-17 22:10:00 │
│ 1414685 ┆ 604323  ┆ 68            ┆ 2026-08-17T22:10:00.000000 ┆ 2026-08-17 22:10:00 │
│ 774398  ┆ 197782  ┆ 240           ┆ 2026-08-17T22:10:00.000000 ┆ 2026-08-17 22:10:00 │
│ 1654055 ┆ 607711  ┆ 1080          ┆ 2026-08-17T22:10:00.000000 ┆ 2026-08-17 22:10:00 │
│ 1534511 ┆ 607543  ┆ 660           ┆ 2026-08-17T22:10:00.000000 ┆ 2026-08-17 22:10:00 │
└─────────┴─────────┴───────────────┴────────────────────────────┴─────────────────────┘


In [21]:
import polars as pl

# Assuming your lazy frame from Step 1 is named `lazy_transit_df`
print("Calculating hourly delay patterns across the network...")

# Group by hour and calculate metrics
hourly_delay_df = (
    lazy_transit_df
    .with_columns(
        pl.col("parsed_timestamp").dt.hour().alias("hour_of_day")
    )
    .group_by("hour_of_day")
    .agg([
        pl.count("trip_id").alias("total_observations"),
        pl.col("delay_seconds").mean().alias("mean_delay_sec"),
        pl.col("delay_seconds").median().alias("median_delay_sec"),
        pl.col("delay_seconds").max().alias("max_delay_sec"),
        # Proportion of delayed trips (delay greater than 60 seconds)
        (pl.col("delay_seconds") > 60).mean().alias("prop_delayed_over_1min")
    ])
    .sort("hour_of_day")
    .collect()
)

print(hourly_delay_df)

Calculating hourly delay patterns across the network...
shape: (24, 6)
┌─────────────┬─────────────────┬────────────────┬────────────────┬───────────────┬────────────────┐
│ hour_of_day ┆ total_observati ┆ mean_delay_sec ┆ median_delay_s ┆ max_delay_sec ┆ prop_delayed_o │
│ ---         ┆ ons             ┆ ---            ┆ ec             ┆ ---           ┆ ver_1min       │
│ i8          ┆ ---             ┆ f64            ┆ ---            ┆ i32           ┆ ---            │
│             ┆ u32             ┆                ┆ f64            ┆               ┆ f64            │
╞═════════════╪═════════════════╪════════════════╪════════════════╪═══════════════╪════════════════╡
│ 0           ┆ 12060           ┆ 90.335655      ┆ 60.0           ┆ 6804          ┆ 0.488806       │
│ 1           ┆ 5667            ┆ 160.378154     ┆ 60.0           ┆ 6804          ┆ 0.46462        │
│ 2           ┆ 2795            ┆ 111.725581     ┆ 60.0           ┆ 4764          ┆ 0.476923       │
│ 3           ┆ 2517

In [22]:
import polars as pl

print("Analyzing stop hotspots across the network...")

# Group by stop_id to find the most delayed stations
stop_hotspots_df = (
    lazy_transit_df
    .group_by("stop_id")
    .agg([
        pl.count("trip_id").alias("total_stop_visits"),
        pl.col("delay_seconds").mean().alias("mean_delay_sec"),
        pl.col("delay_seconds").max().alias("max_delay_sec"),
        # Total cumulative delay experienced at this stop
        pl.col("delay_seconds").sum().alias("cumulative_delay_sec"),
    ])
    .sort("mean_delay_sec", descending=True)
    .head(15)  # Top 15 worst bottleneck stops
    .collect()
)

print(stop_hotspots_df)


Analyzing stop hotspots across the network...
shape: (15, 5)
┌─────────┬───────────────────┬────────────────┬───────────────┬──────────────────────┐
│ stop_id ┆ total_stop_visits ┆ mean_delay_sec ┆ max_delay_sec ┆ cumulative_delay_sec │
│ ---     ┆ ---               ┆ ---            ┆ ---           ┆ ---                  │
│ i64     ┆ u32               ┆ f64            ┆ i32           ┆ i32                  │
╞═════════╪═══════════════════╪════════════════╪═══════════════╪══════════════════════╡
│ 691246  ┆ 13                ┆ 2545.0         ┆ 2545          ┆ 33085                │
│ 107734  ┆ 39                ┆ 2521.692308    ┆ 4200          ┆ 98346                │
│ 606227  ┆ 189               ┆ 1413.883598    ┆ 7125          ┆ 267224               │
│ 606417  ┆ 136               ┆ 1185.397059    ┆ 3442          ┆ 161214               │
│ 690512  ┆ 51                ┆ 1181.607843    ┆ 3045          ┆ 60262                │
│ …       ┆ …                 ┆ …              ┆ …         

What this tells us:


mean_delay_sec: Highlights which stations consistently suffer from structural delays (e.g., due to heavy passenger boarding, tight scheduling, or intersection congestion).

cumulative_delay_sec: Shows where the greatest total impact on passenger time occurs across the 2-week telemetry period.


In [ ]:
import polars as pl
import plotly.express as px

# Assuming `stop_hotspots_df` from Step 3 is already collected into a Polars DataFrame
# If needed, you can collect it first:
# stop_hotspots_df = lazy_transit_df.group_by("stop_id").agg([...]).sort(..., descending=True).head(15).collect()

# Convert to pandas for Plotly compatibility
df_plot = stop_hotspots_df.to_pandas()

# Create an engaging Plotly bar chart
fig = px.bar(
    df_plot,
    x="stop_id",
    y="mean_delay_sec",
    text="total_stop_visits",
    title="Bremen Transit: Top 15 Delay Hotspots (Average Delay in Seconds)",
    labels={"stop_id": "Stop ID", "mean_delay_sec": "Average Delay (Seconds)"},
    template="plotly_white",
    color="mean_delay_sec",
    color_continuous_scale="Reds"
)

fig.update_layout(
    xaxis_type="category",
    xaxis_title="Station / Stop ID",
    yaxis_title="Mean Delay (s)",
    title_font_size=18,
    font=dict(family="Arial", size=12)
)

# Show the interactive chart
fig.show()

# Optional: Save as an interactive HTML file for your presentation
fig.write_html("bremen_transit_hotspots.html")
print("Interactive hotspot visualization saved as 'bremen_transit_hotspots.html'.")

Interactive hotspot visualization saved as 'bremen_transit_hotspots.html'.


In [26]:
from pathlib import Path
import polars as pl

lookup_path = Path("lookup_table.csv")
if not lookup_path.exists():
    lookup_path = Path("./update_data/lookup_table.csv")

if lookup_path.exists():
    print(f"Loading lookup table from: {lookup_path.resolve()}")
    
    # Safely read lookup table with explicit schema overrides for columns with mixed text/numbers
    lookup_df = pl.read_csv(
        lookup_path,
        schema_overrides={
            "route_short_name": pl.String,
            "stop_id": pl.Int64
        },
        infer_schema_length=10000
    )
    
    # Join with your telemetry lazy frame
    enriched_transit_lf = (
        lazy_transit_df
        .join(
            lookup_df.lazy().select(["stop_id", "stop_name", "route_short_name", "trip_headsign"]),
            on="stop_id",
            how="left"
        )
    )

    # Re-run top delay hotspots with human-readable names
    named_stop_hotspots = (
        enriched_transit_lf
        .group_by(["stop_id", "stop_name", "route_short_name"])
        .agg([
            pl.count("trip_id").alias("total_stop_visits"),
            pl.col("delay_seconds").mean().alias("mean_delay_sec"),
            pl.col("delay_seconds").max().alias("max_delay_sec"),
            pl.col("delay_seconds").sum().alias("cumulative_delay_sec"),
        ])
        .sort("mean_delay_sec", descending=True)
        .head(15)
        .collect()
    )

    print(named_stop_hotspots)
else:
    print("lookup_table.csv not found.")

Loading lookup table from: C:\Users\yempa\OneDrive\Attachments\Documents\New folder\Data_visualisation\lookup_table.csv
shape: (15, 7)
┌─────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬─────────────┐
│ stop_id ┆ stop_name    ┆ route_short_ ┆ total_stop_v ┆ mean_delay_s ┆ max_delay_se ┆ cumulative_ │
│ ---     ┆ ---          ┆ name         ┆ isits        ┆ ec           ┆ c            ┆ delay_sec   │
│ i64     ┆ str          ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---         │
│         ┆              ┆ str          ┆ u32          ┆ f64          ┆ i32          ┆ i32         │
╞═════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╪═════════════╡
│ 691246  ┆ Bremen Fried ┆ N9           ┆ 13           ┆ 2545.0       ┆ 2545         ┆ 33085       │
│         ┆ rich-Engels- ┆              ┆              ┆              ┆              ┆             │
│         ┆ Straße       ┆              ┆              ┆ 